# 05 â€” Train 1D CNN (3-class) + saliency

Trains the 1D CNN on the same subject-level splits as the tabular models, on the AHA 3-class label, and computes a per-channel saliency map over the test set so we can say which time regions of the PPG/ECG waveforms drove the model's predictions.

**Prerequisites:**
1. `02_build_features.ipynb` run with `WRITE_SIGNAL_CACHE = True` to produce `signals.h5`.
2. `03_train_tabular.ipynb` run first to write `splits.json` (multi-class stratified).

Outputs:
- `models/cnn_3class.pt` â€” model state dict
- `data/processed/cnn_metrics.json` â€” `MultiClassMetrics`
- `data/processed/cnn_saliency.npy` â€” `(2, 1000)` mean |gradient| over the test loader
- `data/processed/cnn_mean_waveform.npy` â€” `(2, 1000)` mean test-set waveform (backdrop for saliency overlay in nb 04)

CUDA is auto-selected on the 4070; CPU fallback works but will be slow.

In [1]:
EPOCHS = 15
BATCH_SIZE = 256
LR = 1e-3
NUM_WORKERS = 4

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from bme_ml.paths import setup_paths
from bme_ml.splits import Splits, add_subject_id
from bme_ml.labels import add_multiclass_column
from bme_ml.cnn import (
    SegmentDataset, subjects_to_row_indices, CNN1D, TrainConfig, train, predict_proba,
)
from bme_ml.evaluation import evaluate_multiclass
from bme_ml.saliency import compute_saliency, mean_waveform

LABEL_COL = 'label_3class'
N_CLASSES = 3

paths = setup_paths()

if torch.cuda.is_available():
    device = torch.device('cuda')
    print('device: cuda /', torch.cuda.get_device_name(0))
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    NUM_WORKERS = 0
    print('device: mps (Apple Silicon)')
else:
    device = torch.device('cpu')
    print('device: cpu (training will be slow â€” consider lowering EPOCHS)')

device: cuda / NVIDIA GeForce RTX 4070


In [3]:
# Build labels by joining h5 row indices with the parquet (parquet row
# index == h5 row index by construction in pipeline.py).
features = pd.read_parquet(paths.features_parquet)
features = add_subject_id(features)
features = add_multiclass_column(features, col=LABEL_COL)
all_labels = features[LABEL_COL].to_numpy(dtype=np.int64)

splits = Splits.from_json(paths.splits_json)
fold0 = splits.cv_folds[0]
train_subj = list(fold0['train'])
val_subj = list(fold0['val'])
test_subj = splits.test_subjects
print(f'subjects  train={len(train_subj)}  val={len(val_subj)}  test={len(test_subj)}')

train_idx = subjects_to_row_indices(paths.signals_h5, train_subj)
val_idx   = subjects_to_row_indices(paths.signals_h5, val_subj)
test_idx  = subjects_to_row_indices(paths.signals_h5, test_subj)

train_y = all_labels[train_idx]
val_y   = all_labels[val_idx]
test_y  = all_labels[test_idx]
print(f'segments  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')
print('train class counts :', np.bincount(train_y, minlength=N_CLASSES))
print('test  class counts :', np.bincount(test_y, minlength=N_CLASSES))

ds_train = SegmentDataset(paths.signals_h5, train_idx, train_y)
ds_val   = SegmentDataset(paths.signals_h5, val_idx, val_y)
ds_test  = SegmentDataset(paths.signals_h5, test_idx, test_y)

common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
dl_train = DataLoader(ds_train, shuffle=True,  **common)
dl_val   = DataLoader(ds_val,   shuffle=False, **common)
dl_test  = DataLoader(ds_test,  shuffle=False, **common)

subjects  train=7671  val=1918  test=2398


segments  train=81965  val=20492  test=25687
train class counts : [30354 12789 38822]
test  class counts : [ 9335  3966 12386]


In [4]:
# Class weights â€” counter the dataset's hypertension over-representation.
from sklearn.utils.class_weight import compute_class_weight
present_classes = np.unique(train_y)
present_weights = compute_class_weight('balanced', classes=present_classes, y=train_y)
class_weights = np.ones(N_CLASSES, dtype=np.float32)
for c, w in zip(present_classes, present_weights):
    class_weights[int(c)] = float(w)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)
print('class weights:', class_weights)

model = CNN1D(in_channels=2, n_classes=N_CLASSES)
n_params = sum(p.numel() for p in model.parameters())
print(f'CNN parameters: {n_params/1e3:.1f} k')

cfg = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, num_workers=NUM_WORKERS)
model = train(model, dl_train, dl_val, cfg, device, class_weights=class_weights_t)

class weights: [0.900101  2.136341  0.7037676]
CNN parameters: 397.1 k


epoch 01/15  train_loss=0.9299  val_macro_f1=0.5765


epoch 02/15  train_loss=0.8134  val_macro_f1=0.5493


epoch 03/15  train_loss=0.7518  val_macro_f1=0.5527


epoch 04/15  train_loss=0.6937  val_macro_f1=0.6034


...

epoch 11/15  train_loss=0.4067  val_macro_f1=0.6956


epoch 12/15  train_loss=0.3741  val_macro_f1=0.6916


epoch 13/15  train_loss=0.3404  val_macro_f1=0.7140


epoch 14/15  train_loss=0.3180  val_macro_f1=0.7161


epoch 15/15  train_loss=0.3029  val_macro_f1=0.7195


In [5]:
y_true, y_pred, y_proba = predict_proba(model, dl_test, device)
metrics = evaluate_multiclass(y_true, y_pred, y_proba, n_classes=N_CLASSES)
print(metrics)

torch.save(model.state_dict(), paths.models / 'cnn_3class.pt')
(paths.processed / 'cnn_metrics.json').write_text(json.dumps(metrics.__dict__, indent=2))

# Saliency over the test loader â€” use each sample's predicted class so the
# heatmap reflects "what the model focused on for whichever class it chose".
sal = compute_saliency(model, dl_test, device, target='predicted')
mean_wave = mean_waveform(dl_test)
np.save(paths.processed / 'cnn_saliency.npy', sal)
np.save(paths.processed / 'cnn_mean_waveform.npy', mean_wave)
print(f'saved CNN weights, metrics, saliency {sal.shape}, mean waveform {mean_wave.shape}')

MultiClassMetrics(accuracy=0.7341067466033402, f1_macro=0.6859474706757926, confusion=[[6512, 1708, 1115], [661, 2345, 960], [717, 1669, 10000]], hypertensive_auroc=0.9048063455623793, hypertensive_pr_auc=0.8884447152874484, hypertensive_recall=0.8073631519457451, hypertensive_false_negative_rate=0.19263684805425485)


saved CNN weights, metrics, saliency (2, 1000), mean waveform (2, 1000)
